# Đồ án Học máy Nâng cao: Nghiên cứu và Thực nghiệm Mô hình SVM trên tập Iris

## 1. Mục tiêu nghiên cứu
- Khám phá và phân tích đặc trưng dữ liệu (EDA - Exploratory Data Analysis).
- Khảo sát lý thuyết và bản chất toán học của SVM (Support Vector Machine) với các hàm nhân (Kernels: Linear, RBF, Polynomial, Sigmoid).
- Đánh giá sự ảnh hưởng của siêu tham số $C$, $\gamma$, và bậc đa thức `degree` đến biên phân lớp (Decision Boundary) và biên cực đại (Margin).
- Đánh giá chéo k-fold (Cross-Validation), ma trận nhầm lẫn (Confusion Matrix) và phân tích các vectơ hỗ trợ (Support Vectors).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# 1. Tải bộ dữ liệu kinh điển Iris
iris = datasets.load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names
print(f"Kích thước ma trận đặc trưng: {X.shape}")
print(f"Danh sách nhãn: {list(target_names)}")

## 2. Trực quan hóa tương quan đặc trưng (2D Projection)

In [ ]:
# Chọn 2 đặc trưng đại diện: Petal length và Petal width
X_2d = X[:, [2, 3]]

plt.figure(figsize=(8, 5))
for idx, cl in enumerate(np.unique(y)):
    plt.scatter(x=X_2d[y == cl, 0], y=X_2d[y == cl, 1],
                alpha=0.8, label=target_names[cl],
                edgecolor='black', s=45)
plt.xlabel(feature_names[2])
plt.ylabel(feature_names[3])
plt.legend(loc='upper left')
plt.title('Phân bố các loài hoa Iris (Petal Length vs Petal Width)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 3. Khảo sát và Đánh giá các Hàm nhân (Kernel Comparison)

Các hàm nhân được so sánh:
1. **Linear Kernel**: $K(x, z) = x^T z$
2. **RBF Kernel (Gaussian)**: $K(x, z) = \exp(-\gamma ||x - z||^2)$
3. **Polynomial Kernel**: $K(x, z) = (\gamma x^T z + r)^d$
4. **Sigmoid Kernel**: $K(x, z) = \tanh(\gamma x^T z + r)$

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kernels = ['linear', 'rbf', 'poly', 'sigmoid']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('=== ĐÁNH GIÁ 5-FOLD STRATIFIED CROSS VALIDATION ===')
for k in kernels:
    clf = SVC(kernel=k, probability=True, random_state=42)
    scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='accuracy')
    clf.fit(X_scaled, y)
    n_sv = clf.support_vectors_.shape[0]
    print(f"Kernel {k:<8}: Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f}) | Số lượng SVs: {n_sv}")

## 4. Vẽ Ranh giới quyết định (Decision Boundary) và Vectơ hỗ trợ (Support Vectors)

In [ ]:
def plot_decision_boundary(kernel_type, C_val=1.0, gamma_val='scale'):
    scaler_2d = StandardScaler()
    X_2d_scaled = scaler_2d.fit_transform(X_2d)
    model = SVC(kernel=kernel_type, C=C_val, gamma=gamma_val, random_state=42)
    model.fit(X_2d_scaled, y)

    x_min, x_max = X_2d_scaled[:, 0].min() - 0.5, X_2d_scaled[:, 0].max() + 0.5
    y_min, y_max = X_2d_scaled[:, 1].min() - 0.5, X_2d_scaled[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    plt.figure(figsize=(8, 5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    for idx, cl in enumerate(np.unique(y)):
        plt.scatter(x=X_2d_scaled[y == cl, 0], y=X_2d_scaled[y == cl, 1],
                    label=target_names[cl], edgecolor='black')
    # Vẽ các vectơ hỗ trợ (Support Vectors)
    plt.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                s=120, facecolors='none', edgecolors='k', linewidths=1.5, label='Support Vectors')
    plt.title(f'Ranh giới phân lớp SVM (Kernel: {kernel_type}, C={C_val})')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.show()

plot_decision_boundary('linear', C_val=1.0)
plot_decision_boundary('rbf', C_val=1.0)